# Demo: Login → Online → Logout
**Simulador de Planta Móvil — Backend**

Este notebook demuestra el flujo completo de sesión de un usuario:

1. **Login** — el usuario inicia sesión y `online` pasa a `True`
2. **Verificación en BD** — se consulta la contraseña directamente para demostrar que está guardada como hash bcrypt
3. **Logout** — el usuario cierra sesión y `online` vuelve a `False`
4. **GET usuarios** — se listan todos los usuarios y se confirma que el campo `online` cambió

> Ejecutar con el backend corriendo en `http://127.0.0.1:8000`

---
## 1. Configuración inicial

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "backend"))

import requests

BASE_URL = "http://127.0.0.1:8000"

CORREO   = "a@gmail.com"
PASSWORD = "12345678"   # contraseña en texto plano

print("✓ Configuración lista")
print(f"  Base URL : {BASE_URL}")
print(f"  Usuario  : {CORREO}")

✓ Configuración lista
  Base URL : http://127.0.0.1:8000
  Usuario  : a@gmail.com


---
## 2. Estado inicial — GET usuarios (antes del login)

In [2]:
resp = requests.get(f"{BASE_URL}/auth/users")
usuarios = resp.json()["usuarios"]

print(f"Total usuarios: {len(usuarios)}\n")
print(f"{'ID':<5} {'Nombre':<15} {'Correo':<30} {'Online':<8} {'Admin'}")
print("-" * 70)
for u in usuarios:
    marca = " ← objetivo" if u["email"] == CORREO else ""
    print(f"{u['user_id']:<5} {u['name']:<15} {u['email']:<30} {str(u['online']):<8} {u['rol_admin']}{marca}")

Total usuarios: 10

ID    Nombre          Correo                         Online   Admin
----------------------------------------------------------------------
10    Tilina          sologym@correo.com             False    False
7     Prueba1         andres@gmail.com               False    False
8     Yiyi            yiyi1@gmail.com                False    False
9     Yiyi            yiyi@hotmail.com               False    False
5     Iyuqui          iyuquitarako@gmail.com         False    False
6     cristian        a@gmail.com                    False    False ← objetivo
4     Yiyicell        yiyicell26@gmail.com           False    False
1     Ana Gómez       ana@correo.com                 False    False
2     yyyyyy          yyyy@gmail.com                 False    False
3     Yiyi            yiyi@gmail.com                 False    False


---
## 3. Login — `online` debe cambiar a `True`

In [3]:
resp = requests.post(f"{BASE_URL}/auth/login", json={"correo": CORREO, "password": PASSWORD})

print(f"Status HTTP : {resp.status_code}")
data = resp.json()

if resp.status_code == 200:
    u = data["usuario"]
    USER_ID = u["id"]
    print(f"\n✓ Login exitoso")
    print(f"  ID        : {u['id']}")
    print(f"  Nombre    : {u['nombre']}")
    print(f"  Correo    : {u['correo']}")
    print(f"  online    : {u['online']}   ← debe ser True")
    print(f"  rol_admin : {u['rol_admin']}")
    print(f"  creado    : {u['creado']}")
else:
    print(f"\n✗ Error: {data['mensaje']}")
    USER_ID = None

Status HTTP : 200

✓ Login exitoso
  ID        : 6
  Nombre    : cristian
  Correo    : a@gmail.com
  online    : True   ← debe ser True
  rol_admin : False
  creado    : 2026-04-13


---
## 4. Confirmar `online = True` — GET usuarios tras el login

In [4]:
resp = requests.get(f"{BASE_URL}/auth/users")
usuarios = resp.json()["usuarios"]

print(f"{'ID':<5} {'Nombre':<15} {'Correo':<30} {'Online':<8} {'Admin'}")
print("-" * 70)
for u in usuarios:
    marca = " ← debe ser True" if u["email"] == CORREO else ""
    print(f"{u['user_id']:<5} {u['name']:<15} {u['email']:<30} {str(u['online']):<8} {u['rol_admin']}{marca}")

ID    Nombre          Correo                         Online   Admin
----------------------------------------------------------------------
10    Tilina          sologym@correo.com             False    False
9     Yiyi            yiyi@hotmail.com               False    False
7     Prueba1         andres@gmail.com               False    False
8     Yiyi            yiyi1@gmail.com                False    False
6     cristian        a@gmail.com                    True     False ← debe ser True
4     Yiyicell        yiyicell26@gmail.com           False    False
5     Iyuqui          iyuquitarako@gmail.com         False    False
3     Yiyi            yiyi@gmail.com                 False    False
2     yyyyyy          yyyy@gmail.com                 False    False
1     Ana Gómez       ana@correo.com                 False    False


---
## 5. Contraseña en la BD — demostración del hash bcrypt

In [5]:
import psycopg2
from dotenv import load_dotenv

# El .env está en backend/, un nivel arriba de examples/
env_path = os.path.join(os.path.dirname(os.getcwd()), "backend", ".env")
# Si ya estamos en la raíz del proyecto, también lo intentamos
if not os.path.exists(env_path):
    env_path = os.path.join(os.getcwd(), "backend", ".env")

load_dotenv(dotenv_path=env_path)

pg_host = os.getenv("PG_HOST")
print(f"Conectando a: {pg_host}")

conn = psycopg2.connect(
    host=pg_host,
    port=int(os.getenv("PG_PORT", "5432")),
    dbname=os.getenv("PG_DATABASE"),
    user=os.getenv("PG_USER"),
    password=os.getenv("PG_PASSWORD"),
)
cur = conn.cursor()
cur.execute('SELECT name, email, hashed_password FROM "user" WHERE email = %s', (CORREO,))
fila = cur.fetchone()
conn.close()

if fila:
    print(f"Nombre   : {fila[0]}")
    print(f"Correo   : {fila[1]}")
    print(f"\nContraseña almacenada en la BD:")
    print(f"  {fila[2]}")
    es_bcrypt = fila[2].startswith("$2b$") or fila[2].startswith("$2a$")
    print(f"\n→ ¿Es hash bcrypt? {'Sí ✓' if es_bcrypt else 'No ✗'}")
    print(f"→ La contraseña original '{PASSWORD}' nunca se guarda en texto plano.")

Conectando a: 100.68.116.64
Nombre   : cristian
Correo   : a@gmail.com

Contraseña almacenada en la BD:
  $2b$12$ovPNJ7/FY5Tdw/9zHy5kVO6J5xrUe9wSZeScnmY6bXPSwY7KpYwYu

→ ¿Es hash bcrypt? Sí ✓
→ La contraseña original '12345678' nunca se guarda en texto plano.


---
## 6. Logout — `online` debe volver a `False`

In [6]:
if USER_ID:
    resp = requests.post(f"{BASE_URL}/auth/logout/{USER_ID}")
    print(f"Status HTTP : {resp.status_code}")
    print(f"Respuesta   : {resp.json()['mensaje']}")
else:
    print("⚠ No hay USER_ID disponible. Ejecuta primero la celda de login.")

Status HTTP : 200
Respuesta   : Sesión cerrada correctamente.


---
## 7. Estado final — GET usuarios (después del logout)

In [7]:
resp = requests.get(f"{BASE_URL}/auth/users")
usuarios = resp.json()["usuarios"]

print(f"{'ID':<5} {'Nombre':<15} {'Correo':<30} {'Online':<8} {'Admin'}")
print("-" * 70)
for u in usuarios:
    marca = " ← debe ser False" if u["email"] == CORREO else ""
    print(f"{u['user_id']:<5} {u['name']:<15} {u['email']:<30} {str(u['online']):<8} {u['rol_admin']}{marca}")

ID    Nombre          Correo                         Online   Admin
----------------------------------------------------------------------
10    Tilina          sologym@correo.com             False    False
9     Yiyi            yiyi@hotmail.com               False    False
7     Prueba1         andres@gmail.com               False    False
8     Yiyi            yiyi1@gmail.com                False    False
6     cristian        a@gmail.com                    False    False ← debe ser False
4     Yiyicell        yiyicell26@gmail.com           False    False
5     Iyuqui          iyuquitarako@gmail.com         False    False
3     Yiyi            yiyi@gmail.com                 False    False
2     yyyyyy          yyyy@gmail.com                 False    False
1     Ana Gómez       ana@correo.com                 False    False
